<a href="https://colab.research.google.com/github/sejalagnihotri/rag_project/blob/main/prodrag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install chromadb sentence-transformers langchain numpy pandas langchain_text_splitters rank-bm25

In [10]:
!pip install chromadb sentence-transformers langchain rank-bm25 numpy pandas torch langchain_text_splitters

"""
=========================================================
REALISTIC ENTERPRISE RAG PIPELINE
=========================================================

USE CASE:
----------
Financial / Legal / Loan Document Intelligence

FEATURES:
----------
✓ JSON ingestion
✓ Section-aware chunking
✓ Parent-child chunking
✓ Metadata enrichment
✓ Open-source embeddings
✓ Chroma vector DB
✓ BM25 keyword retrieval
✓ Hybrid retrieval
✓ Cross-encoder reranking
✓ Metadata filtering
✓ Query intent parsing
✓ Context compression
✓ Citation generation
✓ Enterprise response formatting

=========================================================
INSTALL
=========================================================

pip install \
chromadb \
sentence-transformers \
langchain \
rank-bm25 \
numpy \
pandas \
torch

=========================================================
PROJECT STRUCTURE
=========================================================

project/
│
├── data/
│   ├── ICE_Memo.json
│   ├── Loan_Agreement.json
│   └── Guarantors_Agreement.json
│
├── chroma_db/
│
└── enterprise_rag.py

"""

# =========================================================
# IMPORTS
# =========================================================

import os
import re
import json
import uuid
import numpy as np

from typing import List, Dict

import chromadb

from rank_bm25 import BM25Okapi

from sentence_transformers import (
    SentenceTransformer,
    CrossEncoder
)

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)


# =========================================================
# CONFIG
# =========================================================

DATA_FOLDER = "./data"

CHROMA_PATH = "./chroma_db"

COLLECTION_NAME = "enterprise_loan_docs"

EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

RERANKER_MODEL = "BAAI/bge-reranker-large"

TOP_K_VECTOR = 10

TOP_K_FINAL = 3

CHUNK_SIZE = 400

CHUNK_OVERLAP = 75


# =========================================================
# DOCUMENT MAP
# =========================================================

DOCUMENT_MAP = {
    "ice memo": "ICE_Memo.pdf",
    "loan agreement": "Loan_Agreement.pdf",
    "guarantors agreement": "Guarantors_Agreement.pdf"
}


# =========================================================
# LOAD MODELS
# =========================================================

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print("Loading reranker model...")

reranker = CrossEncoder(
    RERANKER_MODEL
)


# =========================================================
# CHROMA DB
# =========================================================

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME
)


# =========================================================
# SPLITTERS
# =========================================================

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)


# =========================================================
# SECTION DETECTION
# =========================================================

SECTION_PATTERNS = [
    r"^[A-Z][A-Za-z ]+$",
    r"^\d+\.",
    r"^[A-Z ]{3,}$"
]


def detect_sections(text):

    lines = text.split("\n")

    sections = []

    current_section = "GENERAL"

    buffer = []

    for line in lines:

        line = line.strip()

        is_section = False

        for pattern in SECTION_PATTERNS:

            if re.match(pattern, line):

                is_section = True
                break

        if is_section:

            if buffer:

                sections.append({
                    "section": current_section,
                    "content": "\n".join(buffer)
                })

            current_section = line
            buffer = []

        else:
            buffer.append(line)

    if buffer:

        sections.append({
            "section": current_section,
            "content": "\n".join(buffer)
        })

    return sections


# =========================================================
# TABLE TO TEXT
# =========================================================

def convert_table_to_text(table_data):

    headers = table_data[0]

    rows = []

    for row in table_data[1:]:

        row_text = []

        for h, v in zip(headers, row):

            row_text.append(f"{h}: {v}")

        rows.append(" | ".join(row_text))

    return "\n".join(rows)


# =========================================================
# LOAD DOCUMENT
# =========================================================

def load_document(json_path):

    with open(json_path, "r", encoding="utf-8") as f:

        data = json.load(f)

    document_name = data["file_name"]

    pages = []

    for page in data["pages"]:

        page_text = page.get("text", "")

        table_texts = []

        for table in page.get("tables", []):

            table_texts.append(
                convert_table_to_text(table["data"])
            )

        combined = f"""
{page_text}

{' '.join(table_texts)}
"""

        pages.append({
            "page_number": page["page_number"],
            "content": combined
        })

    return {
        "document_name": document_name,
        "pages": pages
    }


# =========================================================
# PARENT-CHILD CHUNKING
# =========================================================

def create_chunks(document):

    all_chunks = []

    document_name = document["document_name"]

    for page in document["pages"]:

        page_number = page["page_number"]

        sections = detect_sections(
            page["content"]
        )

        for section_data in sections:

            section_name = section_data["section"]

            section_content = section_data["content"]

            # ----------------------------------------
            # PARENT CHUNKS
            # ----------------------------------------

            parent_chunks = parent_splitter.split_text(
                section_content
            )

            for parent_index, parent_chunk in enumerate(parent_chunks):

                parent_id = str(uuid.uuid4())

                # ----------------------------------------
                # CHILD CHUNKS
                # ----------------------------------------

                child_chunks = child_splitter.split_text(
                    parent_chunk
                )

                for child_index, child_chunk in enumerate(child_chunks):

                    child_id = str(uuid.uuid4())

                    enriched_text = f"""
DOCUMENT: {document_name}

SECTION: {section_name}

PAGE: {page_number}

PARENT_CHUNK_ID: {parent_id}

CONTENT:
{child_chunk}
"""

                    all_chunks.append({

                        "id": child_id,

                        "text": enriched_text,

                        "parent_text": parent_chunk,

                        "metadata": {

                            "document_name": document_name,

                            "section_name": section_name,

                            "page_number": page_number,

                            "parent_id": parent_id,

                            "child_index": child_index
                        }
                    })

    return all_chunks


# =========================================================
# BM25 INDEX
# =========================================================

bm25_documents = []

bm25_metadata = []


# =========================================================
# INGEST DOCUMENTS
# =========================================================

def ingest_documents():

    print("\nIngesting documents...\n")

    all_chunks = []

    for file_name in os.listdir(DATA_FOLDER):

        if not file_name.endswith(".json"):
            continue

        print(f"Processing: {file_name}")

        document = load_document(
            os.path.join(DATA_FOLDER, file_name)
        )

        chunks = create_chunks(document)

        all_chunks.extend(chunks)

    print(f"\nTotal Chunks: {len(all_chunks)}")

    # ---------------------------------------------
    # EMBEDDINGS
    # ---------------------------------------------

    texts = [c["text"] for c in all_chunks]

    embeddings = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    ids = []
    docs = []
    metas = []
    embs = []

    for chunk, embedding in zip(all_chunks, embeddings):

        ids.append(chunk["id"])

        docs.append(chunk["text"])

        metas.append(chunk["metadata"])

        embs.append(embedding.tolist())

        # BM25
        bm25_documents.append(
            chunk["text"]
        )

        bm25_metadata.append(chunk)

    collection.add(
        ids=ids,
        documents=docs,
        embeddings=embs,
        metadatas=metas
    )

    print("\nDocuments successfully indexed.\n")


# =========================================================
# BUILD BM25
# =========================================================

def build_bm25():

    tokenized = [
        doc.split(" ")
        for doc in bm25_documents
    ]

    return BM25Okapi(tokenized)


# =========================================================
# QUERY PARSER
# =========================================================

def parse_query(query):

    query_lower = query.lower()

    detected_document = None

    for key, value in DOCUMENT_MAP.items():

        if key in query_lower:

            detected_document = value

            query_lower = query_lower.replace(
                key,
                ""
            )

    return {
        "document_filter": detected_document,
        "clean_query": query_lower.strip()
    }


# =========================================================
# VECTOR SEARCH
# =========================================================

def vector_search(clean_query,
                  document_filter=None):

    query_embedding = embedding_model.encode(
        clean_query,
        normalize_embeddings=True
    ).tolist()

    where_filter = None

    if document_filter:

        where_filter = {
            "document_name": document_filter
        }

    results = collection.query(

        query_embeddings=[query_embedding],

        n_results=TOP_K_VECTOR,

        where=where_filter
    )

    retrieved = []

    for doc, meta in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):

        retrieved.append({
            "text": doc,
            "metadata": meta
        })

    return retrieved


# =========================================================
# BM25 SEARCH
# =========================================================

def bm25_search(query,
                bm25_index,
                top_k=5):

    scores = bm25_index.get_scores(
        query.split(" ")
    )

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in ranked_indices:

        results.append({
            "text": bm25_documents[idx],
            "metadata": bm25_metadata[idx]["metadata"]
        })

    return results


# =========================================================
# HYBRID MERGE
# =========================================================

def merge_results(vector_results,
                  bm25_results):

    unique = {}

    for result in vector_results + bm25_results:

        key = result["text"]

        unique[key] = result

    return list(unique.values())


# =========================================================
# RERANKING
# =========================================================

def rerank(query,
           retrieved_docs):

    pairs = []

    for doc in retrieved_docs:

        pairs.append([
            query,
            doc["text"]
        ])

    scores = reranker.predict(pairs)

    reranked = sorted(
        zip(retrieved_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return reranked[:TOP_K_FINAL]


# =========================================================
# CONTEXT COMPRESSION
# =========================================================

def compress_context(text,
                     query):

    keywords = query.lower().split()

    lines = text.split("\n")

    relevant_lines = []

    for line in lines:

        for keyword in keywords:

            if keyword in line.lower():

                relevant_lines.append(line)

                break

    if not relevant_lines:

        return text[:700]

    return "\n".join(relevant_lines)


# =========================================================
# RESPONSE GENERATION
# =========================================================

def generate_response(query,
                      reranked_results):

    response = []

    response.append("=" * 80)

    response.append(f"QUERY: {query}")

    response.append("=" * 80)

    for idx, (doc, score) in enumerate(reranked_results):

        compressed = compress_context(
            doc["text"],
            query
        )

        metadata = doc["metadata"]

        response.append(f"\nRESULT #{idx+1}")

        response.append(f"Relevance Score: {score:.4f}")

        response.append(
            f"Document: {metadata['document_name']}"
        )

        response.append(
            f"Section: {metadata['section_name']}"
        )

        response.append(
            f"Page: {metadata['page_number']}"
        )

        response.append("\nRelevant Context:\n")

        response.append(compressed)

        response.append("\n" + "-" * 80)

    return "\n".join(response)


# =========================================================
# COMPLETE PIPELINE
# =========================================================

def ask(query,
        bm25_index):

    # ---------------------------------------------
    # STEP 1: PARSE QUERY
    # ---------------------------------------------

    parsed = parse_query(query)

    clean_query = parsed["clean_query"]

    document_filter = parsed["document_filter"]

    # ---------------------------------------------
    # STEP 2: VECTOR SEARCH
    # ---------------------------------------------

    vector_results = vector_search(
        clean_query,
        document_filter
    )

    # ---------------------------------------------
    # STEP 3: BM25 SEARCH
    # ---------------------------------------------

    keyword_results = bm25_search(
        clean_query,
        bm25_index
    )

    # ---------------------------------------------
    # STEP 4: MERGE
    # ---------------------------------------------

    merged = merge_results(
        vector_results,
        keyword_results
    )

    # ---------------------------------------------
    # STEP 5: RERANK
    # ---------------------------------------------

    reranked = rerank(
        clean_query,
        merged
    )

    # ---------------------------------------------
    # STEP 6: RESPONSE
    # ---------------------------------------------

    response = generate_response(
        query,
        reranked
    )

    return response



"""
=========================================================
REALISTIC ENTERPRISE FLOW
=========================================================

JSON DOCUMENTS
    ↓
Section Detection
    ↓
Parent Chunking
    ↓
Child Chunking
    ↓
Metadata Enrichment
    ↓
Embedding Generation
    ↓
ChromaDB Storage
    ↓
BM25 Indexing

=========================================================

USER QUERY
    ↓
Intent Parsing
    ↓
Metadata Filtering
    ↓
Vector Retrieval
    ↓
BM25 Retrieval
    ↓
Hybrid Merge
    ↓
Cross-Encoder Reranking
    ↓
Context Compression
    ↓
Cited Response

=========================================================

WHY THIS IS ENTERPRISE-GRADE
=========================================================

✓ Handles legal/financial docs
✓ Avoids cross-document confusion
✓ Improves exact-term retrieval
✓ Improves ranking precision
✓ Preserves document hierarchy
✓ Supports explainability
✓ Provides citations
✓ Enables scalable production deployment

=========================================================

PRODUCTION IMPROVEMENTS
=========================================================

1. Add FastAPI service layer
2. Async ingestion
3. Redis query cache
4. GPU embeddings
5. Streaming responses
6. Multi-tenant collections
7. Access control
8. Audit logging
9. LangGraph workflows
10. Agentic retrieval

=========================================================

RECOMMENDED PROD STACK
=========================================================

Embedding:
- BAAI/bge-base-en-v1.5

Reranker:
- BAAI/bge-reranker-large

Vector DB:
- Chroma / Qdrant

LLM:
- Llama 3
- DeepSeek
- Mistral

Serving:
- FastAPI
- vLLM
- Docker
- Kubernetes

=========================================================
"""


Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

'\n=========================================================\nREALISTIC ENTERPRISE FLOW\n=========================================================\n\nJSON DOCUMENTS\n    ↓\nSection Detection\n    ↓\nParent Chunking\n    ↓\nChild Chunking\n    ↓\nMetadata Enrichment\n    ↓\nEmbedding Generation\n    ↓\nChromaDB Storage\n    ↓\nBM25 Indexing\n\n=========================================================\n\nUSER QUERY\n    ↓\nIntent Parsing\n    ↓\nMetadata Filtering\n    ↓\nVector Retrieval\n    ↓\nBM25 Retrieval\n    ↓\nHybrid Merge\n    ↓\nCross-Encoder Reranking\n    ↓\nContext Compression\n    ↓\nCited Response\n\n=========================================================\n\nWHY THIS IS ENTERPRISE-GRADE\n=========================================================\n\n✓ Handles legal/financial docs\n✓ Avoids cross-document confusion\n✓ Improves exact-term retrieval\n✓ Improves ranking precision\n✓ Preserves document hierarchy\n✓ Supports explainability\n✓ Provides citations\n✓ Enables scala

In [15]:
if __name__ == "__main__":

    # ---------------------------------------------
    # INGEST
    # ---------------------------------------------

    ingest_documents()

    # ---------------------------------------------
    # BUILD BM25
    # ---------------------------------------------

    bm25_index = build_bm25()

    # ---------------------------------------------
    # TEST QUERIES
    # ---------------------------------------------

    queries = [

        "What is the interest rate in ICE memo?",

        "What are the events of default in loan agreement?",

        "Who are the guarantors in guarantors agreement?",

        "What is the tenure of the loan?",

        "What collateral is provided?"
    ]

    for q in queries:

        print("\n\n")
        print(ask(q, bm25_index))




Ingesting documents...

Processing: Loan_Agreement.json
Processing: Guarantors_Agreement.json
Processing: ICE_Memo.json

Total Chunks: 13


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Documents successfully indexed.




QUERY: What is the interest rate in ICE memo?

RESULT #1
Relevance Score: 0.1065
Document: ICE_Memo.pdf
Section: Purpose Working capital enhancement and equipment procurement
Page: 1

Relevant Context:

DOCUMENT: ICE_Memo.pdf
SECTION: Purpose Working capital enhancement and equipment procurement
Interest Rate 11.25% p.a.

--------------------------------------------------------------------------------

RESULT #2
Relevance Score: 0.0469
Document: ICE_Memo.pdf
Section: The credit team recommends approval of the proposed AWM UIG Term Loan facility subject to execution of standard
Page: 1

Relevant Context:

DOCUMENT: ICE_Memo.pdf
SECTION: The credit team recommends approval of the proposed AWM UIG Term Loan facility subject to execution of standard
Item: Borrower Name | Details: Zenith Agro Industries Pvt. Ltd.
Item: Loan Amount | Details: INR 5,00,00,000
Item: Purpose | Details: Working capital enhancement and equipment procurement
Item: Interest Rate